In [2]:
# Import libraries
import pandas as pd


In [ ]:
# Load data from CSV files

diagnoses_icd = pd.read_csv('/mnt/data/mimic-code/mimic-code/mimic-iv/2.2/hosp/diagnoses_icd.csv')
admissions = pd.read_csv('/mnt/data/mimic-code/mimic-code/mimic-iv/2.2/hosp/admissions.csv')
prescriptions = pd.read_csv('/mnt/data/mimic-code/mimic-code/mimic-iv/2.2/hosp/prescriptions.csv')
discharge = pd.read_csv('/mnt/data/mimic-code/mimic-code/mimic-iv/mimic-iv-note/2.2/note/discharge.csv')
d_icd_diagnoses = pd.read_csv('/mnt/data/mimic-code/mimic-code/mimic-iv/2.2/hosp/d_icd_diagnoses.csv')

print("Data loaded successfully.")

In [ ]:
# Step 2: Filter ICD codes related to diabetes
# Filter ICD codes that mention "diabetes" in their description
diabetes_icd_codes = d_icd_diagnoses[d_icd_diagnoses['long_title'].str.contains('diabetes', case=False, na=False)]
print(diabetes_icd_codes)

In [ ]:

# Step 3: Filter the diagnoses data to include only diabetes-related ICD codes
diabetes_codes = diagnoses_icd[diagnoses_icd['icd_code'].isin(diabetes_icd_codes['icd_code'])]
print(diabetes_codes.head())

In [ ]:
# Add descriptive titles for ICD codes by merging
diabetes_codes = diabetes_codes.merge(diabetes_icd_codes[['icd_code', 'long_title']], on='icd_code', how='left')

# Step 4: Merge with hospital admissions to get hospital details
diabetes_admissions = diabetes_codes.merge(admissions, on='hadm_id', how='inner')
print(diabetes_admissions.head())

In [ ]:
# Step 5: Merge with discharge summaries
# Get the discharge summaries from the 'text' column of the discharge table
diabetes_with_notes = diabetes_admissions.merge(discharge[['hadm_id', 'text']], on='hadm_id', how='left')
diabetes_with_notes.rename(columns={'subject_id_x': 'subject_id'}, inplace=True)
print(diabetes_with_notes.head())

In [ ]:
# Step 6: Filter prescriptions for diabetes-related medications (e.g., insulin, metformin)
diabetes_prescriptions = prescriptions[prescriptions['drug'].str.contains('insulin|metformin', case=False, na=False)]
diabetes_prescriptions.head()
# Merge prescriptions with the diabetes dataset based on subject_id
diabetes_full_data = diabetes_with_notes.merge(diabetes_prescriptions, on='subject_id', how='left')
print(diabetes_full_data.head())

In [ ]:
print(len(diabetes_full_data.columns))

column_names = diabetes_full_data.columns.tolist()
print(column_names)

In [ ]:

# Step 7: Save the final enriched diabetes dataset
diabetes_full_data.to_csv('/mnt/data/andres/diabetes_full_data.csv', index=False)
print("Enriched diabetes data saved to 'diabetes_full_data.csv'.")

In [ ]:
# Select only the required columns
columns_to_keep = [
    'subject_id', 'hadm_id_x', 'icd_code', 'long_title', 'drug', 'text',
    'admittime', 'dischtime', 'admission_type', 'prod_strength', 'route', 'race'
]

diabetes_data = diabetes_full_data[columns_to_keep]

# Save the filtered dataframe to a CSV file
diabetes_data.to_csv('filtered_diabetes_data.csv', index=False)

print("Filtered data saved as 'filtered_diabetes_data.csv'")

In [9]:
diabetes_data = pd.read_csv('filtered_diabetes_data.csv')
# Remove duplicate rows (keeping the first occurrence)
df_no_duplicates = diabetes_data.drop_duplicates()

# Save the cleaned data to a new CSV file
df_no_duplicates.to_csv('filtered_diabetes_data.csv', index=False)
diabetes_data = pd.read_csv('filtered_diabetes_data.csv')


In [14]:
# 1. Preprocessing data
import pandas as pd
import re

# Load the data
diabetes_data = pd.read_csv('filtered_diabetes_data.csv')

In [15]:
# Clean the `long_title` and `text` columns
diabetes_data['long_title'] = diabetes_data['long_title'].str.lower().str.strip()
diabetes_data['text'] = diabetes_data['text'].str.lower().str.strip()

# Remove special characters, if necessary
diabetes_data['long_title'] = diabetes_data['long_title'].apply(lambda x: re.sub(r'[^\w\s]', '', str(x)))
diabetes_data['text'] = diabetes_data['text'].apply(lambda x: re.sub(r'[^\w\s]', '', str(x)))

In [16]:
# Filter rows where long_title or text mentions diabetes
diabetes_data = diabetes_data[
    diabetes_data['text'].str.contains('diabetes', na=False)
]


In [17]:
# Check for exact duplicates
duplicates = diabetes_data[diabetes_data.duplicated()]
print(f"Number of exact duplicate rows: {len(duplicates)}")

# Remove exact duplicates
diabetes_data = diabetes_data.drop_duplicates()

Number of exact duplicate rows: 0


In [18]:
# Check for duplicates based on specific columns
partial_duplicates = diabetes_data[diabetes_data.duplicated(subset=['subject_id', 'icd_code', 'drug', 'text'])]
print(f"Number of partial duplicate rows: {len(partial_duplicates)}")

# Remove partial duplicates
diabetes_data = diabetes_data.drop_duplicates(subset=['subject_id', 'icd_code', 'drug', 'text'])


Number of partial duplicate rows: 252


In [37]:
# Combine columns into a single context column for the LLM
diabetes_data['context'] = diabetes_data.apply(
    lambda row: f"Patient ID: {row['subject_id']} "
                f"Patient Race: {row['race']} "
                f" Admission Type: {row['admission_type']} "
                f" Diagnosis: {row['long_title']} (ICD Code: {row['icd_code']}) "
                f" Medication: {row['drug']} "
                f" Discharge Summary: {row['text']} ",
    axis=1
)
print(diabetes_data)

      subject_id  hadm_id_x icd_code  \
1       10000980   20897796    E1122   
4       10000980   20897796   E11319   
7       10000980   20897796    E1151   
10      10000980   24947999    25000   
13      10000980   25242409    25040   
...          ...        ...      ...   
1075    10030746   22297761     E119   
1076    10030753   20090856    E1010   
1077    10030753   20090856    E1010   
1078    10030753   20090856    E1010   
1083    10030753   20090856    E1010   

                                             long_title  \
1     type 2 diabetes mellitus with diabetic chronic...   
4     type 2 diabetes mellitus with unspecified diab...   
7     type 2 diabetes mellitus with diabetic periphe...   
10    diabetes mellitus without mention of complicat...   
13    diabetes with renal manifestations type ii or ...   
...                                                 ...   
1075     type 2 diabetes mellitus without complications   
1076  type 1 diabetes mellitus with ketoacidosi

In [38]:
# Remove special characters, if necessary
diabetes_data['context'] = diabetes_data['context'].apply(lambda x: re.sub(r'\n', '', str(x)))
diabetes_data['context'] = diabetes_data['context'].apply(lambda x: re.sub(r'___', 'N/A', str(x)))


In [21]:
# # Function to split long text into chunks of 500 words
# def split_text(text, max_words=500):
#     words = text.split()
#     return [' '.join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

# # Apply the function to the text column
# diabetes_data['text_chunks'] = diabetes_data['text'].apply(split_text)

# # Explode the chunks into separate rows
# diabetes_data = diabetes_data.explode('text_chunks').drop(columns=['text']).rename(columns={'text_chunks': 'text'})


In [22]:
# # Check for duplicates after text chunking
# chunk_duplicates = diabetes_data[diabetes_data.duplicated(subset=['subject_id', 'context'])]
# print(f"Number of duplicates after chunking: {len(chunk_duplicates)}")

# # Remove duplicates after chunking
# diabetes_data = diabetes_data.drop_duplicates(subset=['subject_id', 'context'])


Number of duplicates after chunking: 1955


In [39]:
# Ensure uniqueness of patient-context pairs
unique_data = diabetes_data.groupby('subject_id')['context'].apply(lambda x: ' '.join(set(x))).reset_index()

# Rename and verify
unique_data.columns = ['subject_id', 'context']
print(f"Number of unique patient-context pairs: {len(unique_data)}")


Number of unique patient-context pairs: 75


In [40]:
import json
# Save the data
# diabetes_data[['subject_id', 'context']].to_json('diabetes_data_preprocessed.json', orient='records', lines=True)
# Select the desired columns
selected_data = diabetes_data[['context']]

# Convert DataFrame to a list of dictionaries
data_list = selected_data.to_dict('records')

# Convert the list of dictionaries to a JSON string
json_string = json.dumps(data_list)

# Write the JSON string to a file
with open('diabetes_data_preprocessed.json', 'w') as f:
    f.write(json_string)
